
# 🔄 Prefect — Orchestration Tool Notes
> **Topic:** Pipeline Orchestration with Prefect
> **Related tools:** Airflow, Prefect, Dagster (all do the same job — different styles)

---

## 1. 🎯 Why Orchestration Tools Exist

When you run data pipelines in production, things **will** fail — bad data, network issues, API timeouts, cloud hiccups. You need a system that:

- **Auto-retries** failed pipelines (3-4 times before giving up)
- **Sends alerts** if it still fails after retries
- **Tracks** how many pipelines you have, when they ran, how long they took
- **Gives visibility** — a UI to see what passed, failed, is running

```
Without orchestration:          With orchestration (Prefect):
Pipeline fails → silence 😶     Pipeline fails → retry → retry → alert 🔔
No visibility                   Full UI dashboard
Manual debugging                Automatic logs + history
```

### Popular Orchestration Tools:

| Tool | Style | Best for |
|---|---|---|
| **Apache Airflow** | DAG-based, heavy, mature | Large enterprise setups |
| **Prefect** | Python-native, lightweight, modern | Python-first teams, easier to set up |
| **Dagster** | Asset-based, strongly typed | Data asset focused pipelines |

---

## 2. 🌊 Flow — The Core Concept

> **A Flow = the blueprint/recipe for your entire data workflow**

### Real World Analogy:
Think of managing a restaurant kitchen:
- **Flow** = the entire cooking process for one dish
- You coordinate: prep → cooking → plating → serving
- Steps happen in the right order
- You handle what happens if something goes wrong (burns, spills)

### Technically:
A Flow is just a **regular Python function** with the `@flow` decorator added on top. That one decorator gives it superpowers.

```python
# Regular Python function — no tracking, no retries, nothing
def my_function():
    return "Hello"

# Prefect Flow — same function, now fully tracked + managed
from prefect import flow

@flow
def my_flow():
    return "Hello"
```

---

## 3. ✨ 4 Key Characteristics of Flows

### 1️⃣ Flows are Just Python Functions
```python
from prefect import flow

@flow
def my_flow():
    print("This is a Prefect Flow!")
    return "Done"
```
No special syntax to learn — if you know Python, you know how to write a Flow.

---

### 2️⃣ Flows Can Call Tasks (Other Functions)
```python
from prefect import flow, task

@task
def task_a():
    return 10

@task
def task_b():
    return 20

@flow
def parent_flow():
    result1 = task_a()   # calls a task
    result2 = task_b()   # calls another task
    return result1 + result2  # → 30
```
> Flow = orchestrator | Tasks = the actual work units

---

### 3️⃣ Flows Track Execution Automatically
When a Flow runs, Prefect automatically records:

| What | Details |
|---|---|
| ✅ Start time | When did it begin |
| ✅ Each step | Which tasks ran, in what order |
| ✅ Logs | What happened inside each task |
| ✅ End time | When did it finish |
| ✅ Status | Success / Failed / Crashed |
| ✅ UI display | All of this visible in Prefect dashboard |

Zero extra code needed — it's all automatic.

---

### 4️⃣ Flows Can Have Parameters (Makes Them Flexible)
```python
from prefect import flow

@flow
def data_pipeline(source: str, date: str, batch_size: int = 100):
    """
    Parameterized flow — run the same flow with different inputs
    """
    data = extract_data(source, date)
    process_data(data, batch_size)

# Run with different parameters each time
data_pipeline(source="customers", date="2024-01-01", batch_size=500)
data_pipeline(source="orders",    date="2024-01-02", batch_size=200)
```

**Why this matters in production:** Same pipeline logic, different dates/sources/configs — no code duplication.

---

## 4. ⚙️ Flow Decorator Options

```python
from prefect import flow
from datetime import timedelta

@flow(
    name="Customer Data Pipeline",           # display name in UI
    description="Ingests customer data",     # shown in UI
    retries=3,                               # retry 3 times on failure
    retry_delay_seconds=60,                  # wait 60s between retries
    timeout_seconds=3600,                    # fail if takes more than 1hr
    log_prints=True                          # automatically log all print() statements
)
def customer_pipeline():
    pass
```

| Option | What it does |
|---|---|
| `name` | Display name shown in Prefect UI |
| `description` | Describes what the flow does |
| `retries` | How many times to retry on failure |
| `retry_delay_seconds` | How long to wait between retries |
| `timeout_seconds` | Max allowed run time |
| `log_prints` | Auto-captures all print statements as logs |

---

## 5. 🚀 What Happens When You Run a Flow?

```python
if __name__ == "__main__":
    my_flow()  # This triggers the flow
```

### Behind the scenes — step by step:

```
1. Prefect creates a "Flow Run"
        ↓ (unique execution instance — like a job run ID)
2. Assigns it a Run ID
        ↓ (like a tracking number — e.g. "abc-123-xyz")
3. Records start time
        ↓
4. Executes your code
        ↓
5. Tracks each task inside the flow
        ↓
6. Records end time + final status
        ↓  (Success ✅ / Failed ❌ / Crashed 💥)
7. Stores everything in Prefect database
        ↓
8. Displays in Prefect UI dashboard
```

**Every run is isolated** — if run 1 fails and run 2 succeeds, they're tracked separately. Full history always available.

---

## 6. 🔁 Retries — Production Pattern

```python
from prefect import flow, task

# Task level retries (most common)
@task(retries=3, retry_delay_seconds=30)
def fetch_data_from_api():
    # If API fails, retries 3 times, waiting 30s each time
    response = requests.get("https://api.example.com/data")
    return response.json()

# Flow level retries
@flow(retries=2, retry_delay_seconds=60)
def my_pipeline():
    data = fetch_data_from_api()
    process(data)
```

### Retry Flow in Production:
```
Run 1 → FAILS
        ↓ wait 30 seconds
Run 2 → FAILS
        ↓ wait 30 seconds
Run 3 → FAILS
        ↓ wait 30 seconds
Run 4 → FAILS
        ↓
Send ALERT 🔔 (Slack / Email / PagerDuty)
```

**⚠️ Production Tip:** Set retries at the **task level** for things that can fail temporarily (API calls, DB connections). Set retries at the **flow level** for full pipeline-level failures. Don't set both to high numbers — you could end up waiting hours before getting an alert.

---

## 📌 Quick Recap

```
Orchestration Tools  → Airflow, Prefect, Dagster — manage pipelines, retries, alerts
Why needed           → Auto-retry failures, send alerts, track all pipeline runs

Flow                 → Blueprint for entire workflow (@flow decorator on Python function)
Task                 → Individual unit of work inside a flow (@task decorator)

Flow characteristics:
  → Just Python functions with @flow
  → Can call tasks inside
  → Auto-tracks everything (start, steps, logs, end, status)
  → Supports parameters for flexibility

Flow run lifecycle:
  → Run ID → Start → Execute → Track tasks → End → Store → Show in UI

Retries              → Set retries=3, retry_delay_seconds=30 on @task or @flow
```

---

---
---

# 📌 NEW SECTION STARTS HERE — Tasks & Assets
> **Day 2 additions:** Tasks (atomic units), Asset tracking & lineage

---

## 7. ⚙️ Tasks — Atomic Units of Work

> **Task = a single, focused unit of work inside a Flow**

Tasks are:
- **Cacheable** — skip re-running if result already exists
- **Retryable** — auto-retry on failure independently
- **Concurrent** — can run in parallel with other tasks
- **Transactional** — each has a clear success/failure state

### What Prefect Tracks Automatically per Task:
| What | Details |
|---|---|
| Runtime | How long it took |
| Final state | Success / Failed / Crashed / Cached |
| Every state transition | Start → Running → Complete — all recorded |
| Observability | All visible in UI with full logs |

---

## 8. 🚀 Ways to Run Tasks

### Standard Call (sequential):
```python
from prefect import flow, task

@task
def extract():
    return [1, 2, 3]

@task
def transform(data):
    return [x * 2 for x in data]

@flow
def pipeline():
    data = extract()       # runs, waits for result
    result = transform(data)  # runs after extract finishes
```

---

### `.submit()` — Non-blocking (concurrent):
```python
@flow
def pipeline():
    # submit() returns a PrefectFuture immediately — doesn't wait
    future1 = extract.submit()
    future2 = extract.submit()   # both run concurrently!

    # .result() blocks until the future is done
    result1 = future1.result()
    result2 = future2.result()
```

---

### `.map()` — Run same task over a list (parallel):
```python
@task
def process(item):
    return item * 2

@flow
def pipeline():
    items = [1, 2, 3, 4, 5]

    # runs process() for each item — concurrently!
    results = process.map(items)
    return results
```

---

### `.wait()` — Wait without getting result:
```python
@flow
def pipeline():
    future = some_task.submit()
    future.wait()   # waits for completion but doesn't return value
```

---

### Delaying a Task:
```python
from datetime import timedelta

@task
def delayed_task():
    print("Running after delay")

@flow
def pipeline():
    delayed_task.with_options(delay=timedelta(seconds=30)).submit()
    # task will wait 30 seconds before starting
```

---

## 9. 🔀 Task Orchestration Models

| Model | What it means | Use case |
|---|---|---|
| **Client-side orchestration** | Flow logic runs on your machine/server, Prefect just tracks | Most common default |
| **State dependencies** | Task B only runs if Task A succeeded | Conditional pipelines |
| **Background tasks** | Tasks submitted to run in background without blocking flow | Fire-and-forget async work |

### State Dependencies Example:
```python
from prefect import flow, task
from prefect.futures import wait

@task
def task_a():
    return "done"

@task
def task_b():
    return "also done"

@flow
def pipeline():
    future_a = task_a.submit()
    # task_b only runs after task_a completes
    future_b = task_b.submit(wait_for=[future_a])
```

**⚠️ Production Tip:** Use `.submit()` + `.map()` whenever tasks are independent of each other — they'll run in parallel and cut your pipeline runtime significantly. Only use sequential calls when one task genuinely depends on another's output.

---

## 10. 📦 Assets — Track What Your Pipeline Produces

> **Asset = an output or data artifact produced by your pipeline**

Prefect Assets shift focus from *"did the task run?"* to *"is my data healthy and up to date?"*

### Why Assets Matter:
```
Without assets:  You know the pipeline ran ✅
With assets:     You know the DATA is fresh, healthy and where it came from ✅✅
```

---

## 11. 🔑 Core Asset Concepts

### Asset Key (URI):
Every asset has a unique identifier — a URI pointing to where the data lives:
```
s3://my-bucket/bronze/customers/
postgres://my-db/public/orders
adls://gizmobox/silver/customers/
```

### 3 Asset States:

| State | Meaning |
|---|---|
| **Materialized** | Created or updated by a workflow — data exists and is fresh |
| **Referenced** | Used as input by another workflow — it's a dependency |
| **External** | Exists outside Prefect but tracked as a dependency |

---

## 12. ✅ Materializations — Creating Assets

```python
from prefect import flow
from prefect.assets import materialize, Asset

# Define the asset
customers_asset = Asset(
    key="s3://gizmobox/silver/customers/",
    name="Silver Customers",
    description="Cleaned and validated customer data",
    owners=["data-team@company.com"]
)

# Materialize it — create/update the asset
@flow
@materialize(customers_asset)
def build_silver_customers():
    # your transformation logic here
    df = spark.read.table("gizmobox.bronze.customers")
    df_clean = df.dropDuplicates().filter("customer_id IS NOT NULL")
    df_clean.write.format("delta").mode("overwrite") \
        .saveAsTable("gizmobox.silver.customers")
    
    # Add metadata about this materialization
    return {
        "row_count": df_clean.count(),
        "processed_at": "2024-01-01"
    }
```

---

## 13. 🔗 Asset References & Dependencies

```python
from prefect.assets import Asset

# Asset A — upstream
bronze_customers = Asset(key="s3://gizmobox/bronze/customers/")

# Asset B — depends on Asset A
silver_customers = Asset(
    key="s3://gizmobox/silver/customers/",
    asset_deps=[bronze_customers]   # explicit dependency declaration
)
```

Prefect automatically builds the dependency graph — you can see it in the UI.

---

## 14. 📊 Asset Metadata

Add rich metadata during execution — tracked automatically:

```python
@materialize(customers_asset)
def build_customers():
    df = transform_customers()
    
    # Return metadata — stored with the asset
    return {
        "row_count": df.count(),
        "processing_time_seconds": 45,
        "data_quality_score": 0.98,
        "source": "operational-db",
        "null_rate": 0.02
    }
```

---

## 15. 🏥 Asset Health Monitoring

| Color | Status | Meaning |
|---|---|---|
| 🟢 Green | Success | Last materialization succeeded — data is fresh |
| 🔴 Red | Failed | Last materialization failed — data may be stale |
| ⚫ Gray | Not materialized | Asset defined but never run yet |

---

## 16. 📡 Asset Events

Prefect automatically emits events whenever asset actions happen — enabling automation:

| Event | When it fires |
|---|---|
| **Materialization success** | Asset created/updated successfully |
| **Materialization failure** | Asset creation failed |
| **Reference event** | Asset used as input by another workflow |

Use these events to trigger alerts, downstream workflows, or monitoring dashboards.

---

## 17. 🗂️ Asset Organization in UI

Assets are automatically grouped by URI scheme and path:
```
s3://
  └── gizmobox/
        ├── bronze/
        │     └── customers      🟢
        ├── silver/
        │     └── customers      🟢
        └── gold/
              └── sales_summary  🔴  ← failed, needs attention
```

Supports search, filtering and hierarchical browsing in Prefect UI.

---

## 📌 Tasks & Assets — Quick Recap

```
Tasks:
  → Atomic, cacheable, retryable units of work
  → .submit()  → non-blocking, returns Future
  → .map()     → run task over list in parallel
  → .result()  → block and get value from Future
  → .wait()    → block without getting value
  → State deps → task_b.submit(wait_for=[future_a])

Assets:
  → Outputs produced by your pipeline (S3, Delta table, DB table)
  → Key = unique URI identifying the data location
  → 3 states: Materialized / Referenced / External
  → @materialize decorator → creates/updates asset + records metadata
  → asset_deps → declare dependencies between assets
  → Health: 🟢 success / 🔴 failed / ⚫ not yet run
  → Events fired on every materialization → use for alerts & automation
```

---

*More Prefect concepts to be added — Deployments, Schedules, Alerts, Workers*